In [1]:
import cupy as cp
import cv2
import numpy as np
import PIL.Image as Image
import rasterio
import torch
from nvidia import nvimgcodec

NV_IMAGE_DECODE_ENABLED = True
# if the loader has prefetch_factor > 0
# gpu decode will take more gpu memory
# decoder
nvimg_gpu_dec_backends = dict(
    backends=[
        nvimgcodec.Backend(nvimgcodec.GPU_ONLY, load_hint=0.5),
        nvimgcodec.Backend(nvimgcodec.HYBRID_CPU_GPU),
    ]
)
nvimg_cpu_dec_backends = dict(
    backend_kinds=[nvimgcodec.CPU_ONLY],
    # backends=[nvimgcodec.Backend(nvimgcodec.CPU_ONLY)],
)

In [2]:
path = "/home/user/zihancao/Project/hyperspectral-1d-tokenizer/data/miniFrance/tmp/13-2014-0795-6265-LA93-0M50-E080-jp2_patch-0.img.jpg"
# path = "/home/user/zihancao/Project/hyperspectral-1d-tokenizer/data/HyperGlobal/tmp/EO1-part1-EO1H0010262012256110K9_1.img.tiff"

# read using rasterio
# with rasterio.open(path) as src:
#     img_rasterio = src.read().transpose(1, 2, 0)  # HWC
#     print(img_rasterio.shape, img_rasterio.dtype)


cpu_decoder = nvimgcodec.Decoder(**nvimg_cpu_dec_backends)
gpu_decoder = nvimgcodec.Decoder(**nvimg_gpu_dec_backends)

In [ ]:
%%time
img_tifffile = tifffile.imread(path)
print(img_tifffile.shape, img_tifffile.dtype)

In [8]:
%%time
img_nv = gpu_decoder.read(path)
print(img_nv.__cuda_array_interface__)

{'shape': (1024, 1024, 3), 'strides': None, 'typestr': '|u1', 'data': (12921602048, False), 'version': 3, 'stream': 1}
CPU times: user 15.6 ms, sys: 33 μs, total: 15.7 ms
Wall time: 14.9 ms


In [4]:
%%time
img_cv2 = cv2.imread(path)
print(img_cv2.shape, img_cv2.dtype)

(1024, 1024, 3) uint8
CPU times: user 25.1 ms, sys: 8.02 ms, total: 33.2 ms
Wall time: 32.4 ms


In [9]:
%%time
img_pil = Image.open(path)
img_np_from_pil = np.asarray(img_pil)

CPU times: user 42.4 ms, sys: 16 ms, total: 58.5 ms
Wall time: 57 ms


In [ ]:
# cupy
img_cp = cp.asarray(img_nv)
print(img_cp.shape)

# numpy
img_np = cp.asnumpy(img_cp)
print(img_np.shape, img_np.dtype)

In [ ]:
# plot
import PIL.Image as Image

Image.fromarray(img_np)

In [10]:
# directly to numpy
img_np = np.asarray(img_nv.cpu())

# tensor
img_torch = torch.as_tensor(img_nv)
print(img_torch.shape, img_torch.device)

torch.Size([1024, 1024, 3]) cuda:0


In [16]:
# encode

params = nvimgcodec.EncodeParams(quality_type=nvimgcodec.QualityType.LOSSLESS)
encoder = nvimgcodec.Encoder()

img_nv_from_th = nvimgcodec.as_image(img_torch).cuda()
encoded_bytes = encoder.encode(img_nv_from_th, "png", params)
print(f"encoded bytes length: {len(encoded_bytes)}")


# decode again
# decode_params = nvimgcodec.DecodeParams(allow_any_depth=True)
img_nv_decoded = cpu_decoder.decode(encoded_bytes).cpu()
img_np_decoded = np.asarray(img_nv_decoded)
print(img_np_decoded.shape, img_np_decoded.dtype)

encoded bytes length: 2330634
(1024, 1024, 3) uint8


In [ ]:
# plot agian
import PIL.Image as Image

Image.fromarray(img_np_decoded)